In [42]:
import os
import pandas as pd
from datetime import datetime
import json
import gc


# folder_path_demand = '/home/admin1/Downloads/punjab-data-prod-analysis/zirakpur/output_demand/'
folder_path_demanddetails = '/home/prerna/Punjab/punjab-data-prod-analysis/phagwara/output_demand_detail/'

# read active properties & needed columns
property_df = pd.read_csv(
    '/home/prerna/Punjab/punjab-data-prod-analysis/phagwara/eg_pt_property_phagwara.csv',
    usecols=['id', 'propertyid', 'tenantid', 'createdtime', 'additionaldetails', 'ownershipcategory', 'status', 'usagecategory', 'landarea']
)
property_df = property_df[property_df['status'] == 'ACTIVE'].copy()
property_df = property_df[property_df['propertyid'] == 'PT-1014-1010345'].copy()

# read units
# unit_df = pd.read_csv(
#     '/home/prerna/Punjab/punjab-data-prod-analysis/phagwara/eg_pt_unit_phagwara.csv',
#     usecols=['propertyid', 'occupancytype']
# )

address_df = pd.read_csv(
    '/home/prerna/Punjab/punjab-data-prod-analysis/phagwara/eg_pt_address_phagwara.csv',
    usecols=['propertyid', 'locality']
)

owner_df = pd.read_csv(
    '/home/prerna/Punjab/punjab-data-prod-analysis/phagwara/eg_pt_owner_phagwara.csv',
    usecols=['propertyid', 'userid']
)




# read demand
demand_df = pd.read_csv(
    '/home/prerna/Punjab/punjab-data-prod-analysis/phagwara/eg_demand_phagwara.csv',
    dtype={"consumercode": str, "taxamount" : float, "collectionamount" : float},
    low_memory=False,
    usecols=['id', 'taxperiodfrom', 'taxperiodto', 'consumercode', 'status', 'businessservice']
)
demand_df = demand_df[demand_df['status'] == 'ACTIVE'].copy()
demand_df = demand_df[demand_df['businessservice'] == 'PT'].copy()
demand_df = demand_df[demand_df['consumercode'] == 'PT-1014-1010345'].copy()

# read demand (memory‑efficient, in chunks)
# all_chunks = []
# needed_cols = ['id', 'taxperiodfrom', 'taxperiodto', 'consumercode', 'status', 'businessservice']
# for filename in os.listdir(folder_path_demand):
#     if filename.endswith('.csv'):
#         file_path = os.path.join(folder_path_demand, filename)
#         print(f'Loading: {file_path}')
#         chunk = pd.read_csv(file_path, usecols=needed_cols)
#         all_chunks.append(chunk)
# demand_df = pd.concat(all_chunks, ignore_index=True)
# demand_df = demand_df[demand_df['status'] == 'ACTIVE'].copy()
# demand_df = demand_df[demand_df['businessservice'] == 'PT'].copy()
# del all_chunks; gc.collect()

# read demand details (memory‑efficient, in chunks)
all_chunks = []
needed_cols = ['demandid', 'taxamount', 'collectionamount', 'taxheadcode']
for filename in os.listdir(folder_path_demanddetails):
    if filename.endswith('.csv'):
        file_path = os.path.join(folder_path_demanddetails, filename)
        print(f'Loading: {file_path}')
        chunk = pd.read_csv(file_path, usecols=needed_cols)
        all_chunks.append(chunk)
demand_details_df = pd.concat(all_chunks, ignore_index=True)
del all_chunks; gc.collect()

print("✅ Loaded data")

Loading: /home/prerna/Punjab/punjab-data-prod-analysis/phagwara/output_demand_detail/output_4.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/phagwara/output_demand_detail/output_34.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/phagwara/output_demand_detail/output_38.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/phagwara/output_demand_detail/output_6.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/phagwara/output_demand_detail/output_60.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/phagwara/output_demand_detail/output_44.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/phagwara/output_demand_detail/output_17.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/phagwara/output_demand_detail/output_22.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/phagwara/output_demand_detail/output_59.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/phagwara/output_demand_detail/output_29.csv
Loading: /ho

In [43]:
demand_df.to_csv("demand.csv", index=False)

In [44]:
defaulter_property_ids = pd.read_csv(
    '/home/prerna/Punjab/Phagwara_updated_defaulter_report_egov_2.csv',
    usecols=['Property ID', 'IsDefaulter']
)
defaulter_property_ids = defaulter_property_ids[defaulter_property_ids['IsDefaulter'] == 'Yes'].copy()

print(defaulter_property_ids.shape)

# Filter properties that are in defaulters list
filtered_property_df = property_df[property_df['propertyid'].isin(defaulter_property_ids['Property ID'])].copy()

print(filtered_property_df.shape)
print(filtered_property_df.head())

del property_df,defaulter_property_ids; gc.collect()


(10191, 2)
(1, 9)
                                         id       propertyid     tenantid  \
18574  0a67ad40-187e-48cb-95d3-ee49fec27af0  PT-1014-1010345  pb.phagwara   

       status       ownershipcategory usagecategory  landarea    createdtime  \
18574  ACTIVE  INDIVIDUAL.SINGLEOWNER   RESIDENTIAL     250.0  1608802705782   

      additionaldetails  
18574               NaN  


0

In [34]:
print(len(filtered_property_df))
print(len(address_df))             # number of rows in units
print(len(owner_df))  
print(len(demand_df))   # number of rows in demand details
print(len(demand_details_df))   # number of rows in demand details

1
40696
39589
12
3248706


In [45]:
# join pt and unit
joined_pt_owner = filtered_property_df.merge(owner_df, left_on='id', right_on='propertyid', how='left', suffixes=('_property', '_owner'))
del filtered_property_df, owner_df; gc.collect()
joined_pt_owner.head()
print(joined_pt_owner['id'].nunique())

1


In [29]:
joined_pt_owner.head()

,id,propertyid_property,tenantid,status,ownershipcategory,usagecategory,landarea,createdtime,additionaldetails,propertyid_owner,userid
0,0a67ad40-187e-48cb-95d3-ee49fec27af0,PT-1014-1010345,pb.phagwara,ACTIVE,INDIVIDUAL.SINGLEOWNER,RESIDENTIAL,250.0,1608802705782,NaN,0a67ad40-187e-48cb-95d3-ee49fec27af0,00ec4b5b-0e0b-4716-aea1-fb964b42139f


In [46]:
# join demand and demand details
joined_demand = demand_df.merge(demand_details_df, left_on='id', right_on='demandid', how='left', suffixes=('_demand', '_detail'))
print(joined_demand['id'].nunique())
del demand_details_df, demand_df; gc.collect()
joined_demand.head()
joined_demand.to_csv("demand.csv", index=False)

12


In [41]:
# Calculate sums per demand id
# paid_df = (
#     joined_demand.groupby('id', as_index=False)
#     .agg({
#         'consumercode': 'first',  # keep consumercode value
#         'taxamount': 'sum',
#         'collectionamount': 'sum'
#     })
# )

# # Add Paid flag based on difference
# paid_df['Paid'] = paid_df.apply(
#     lambda row: 'Yes' if (row['taxamount'] - row['collectionamount']) <= 0 else 'No',
#     axis=1
# )

# # Merge back to original DF so every row has the Paid column
# joined_demand = joined_demand.merge(
#     paid_df[['id', 'Paid']],
#     on='id',
#     how='left'
# )

# print(joined_demand.head())

# import pytz

# # --- Step 1: Calculate due amounts per demand id ---
# due_df = (
#     joined_demand.groupby('id', as_index=False)
#     .agg({
#         'consumercode': 'first',
#         'taxamount': 'sum',
#         'collectionamount': 'sum'
#     })
# )

# # Actual due amount = tax - collected
# due_df['due_amount'] = due_df['taxamount'] - due_df['collectionamount']

# # Merge back due_amount into demand rows
# joined_with_due = joined_demand.merge(
#     due_df[['id', 'due_amount']],
#     on='id',
#     how='left'
# )

# # --- Step 2: Add FY column ---
# joined_with_due['taxperiodfrom_dt'] = pd.to_datetime(joined_with_due['taxperiodfrom'], unit='ms', utc=True)
# ist = pytz.timezone('Asia/Kolkata')
# joined_with_due['taxperiodfrom_dt'] = joined_with_due['taxperiodfrom_dt'].dt.tz_convert(ist)

# def get_fy(date):
#     if date.month >= 4:
#         fy_start = date.year
#         fy_end = date.year + 1
#     else:
#         fy_start = date.year - 1
#         fy_end = date.year
#     return f"{fy_start}-{str(fy_end)[-2:]}"

# joined_with_due['fy'] = joined_with_due['taxperiodfrom_dt'].apply(get_fy)

# # --- Step 3: Aggregate due per property (consumercode) & FY ---
# due_status_df = (
#     joined_with_due.groupby(['consumercode', 'fy'])['due_amount']
#     .sum()
#     .reset_index()
# )

# # --- Step 4: Pivot FYs into separate columns (actual amounts) ---
# pivot_df = due_status_df.pivot(index='consumercode', columns='fy', values='due_amount').reset_index()

# print(pivot_df.head())



import pytz

# --- Step 1: Add FY column directly on joined_demand ---
joined_demand['taxperiodfrom_dt'] = pd.to_datetime(joined_demand['taxperiodfrom'], unit='ms', utc=True)
ist = pytz.timezone('Asia/Kolkata')
joined_demand['taxperiodfrom_dt'] = joined_demand['taxperiodfrom_dt'].dt.tz_convert(ist)

def get_fy(date):
    if date.month >= 4:
        fy_start = date.year
        fy_end = date.year + 1
    else:
        fy_start = date.year - 1
        fy_end = date.year
    return f"{fy_start}-{str(fy_end)[-2:]}"

joined_demand['fy'] = joined_demand['taxperiodfrom_dt'].apply(get_fy)

# --- Step 2: Aggregate directly at consumercode + FY ---
due_status_df = (
    joined_demand.groupby(['consumercode', 'fy'])
    .agg({
        'taxamount': 'sum',
        'collectionamount': 'sum'
    })
    .reset_index()
)

# --- Step 3: Compute total due = sum(tax) - sum(collection) ---
print(due_status_df['taxamount'])
print(due_status_df['collectionamount'])

due_status_df['due_amount'] = due_status_df['taxamount'].astype(float) - due_status_df['collectionamount'].astype(float)

# --- Step 4: Pivot FYs into columns ---
pivot_df = due_status_df.pivot(index='consumercode', columns='fy', values='due_amount').reset_index()

print(pivot_df.head())


0     1256.0
1     1147.0
2     1038.0
3      929.0
4      819.0
5      618.0
6     1085.0
7     1019.0
8      943.0
9      751.0
10    1424.0
Name: taxamount, dtype: float64
0     1256.00
1     1147.00
2     1038.00
3      929.00
4      819.00
5      618.00
6     1085.00
7     1019.00
8      943.00
9      750.34
10       0.00
Name: collectionamount, dtype: float64
fy     consumercode  2015-16  2016-17  2017-18  2018-19  2019-20  2020-21  \
0   PT-1014-1010345      0.0      0.0      0.0      0.0      0.0      0.0   

fy  2021-22  2022-23       2023-24  2024-25  2025-26  
0       0.0      0.0 -1.136868e-13     0.66   1424.0  


In [9]:
# no_paid_ids = paid_df.loc[paid_df['Paid'] == 'No', 'id']
# print(no_paid_ids)

In [10]:
# import pytz
# # Step 1: Convert to datetime and add FY column
# joined_demand['taxperiodfrom_dt'] = pd.to_datetime(joined_demand['taxperiodfrom'], unit='ms', utc=True)

# # Convert to IST (Asia/Kolkata)
# ist = pytz.timezone('Asia/Kolkata')
# joined_demand['taxperiodfrom_dt'] = joined_demand['taxperiodfrom_dt'].dt.tz_convert(ist)

# def get_fy(date):
#     if date.month >= 4:
#         fy_start = date.year
#         fy_end = date.year + 1
#     else:
#         fy_start = date.year - 1
#         fy_end = date.year
#     return f"{fy_start}-{str(fy_end)[-2:]}"

# joined_demand['fy'] = joined_demand['taxperiodfrom_dt'].apply(get_fy)

# # Step 2: Determine Yes/No/Partial for each consumercode & FY
# def fy_paid_status(values):
#     unique_vals = set(values)
#     if len(unique_vals) > 1:
#         return "Partial"
#     else:
#         return list(unique_vals)[0]

# status_df = (
#     joined_demand.groupby(['consumercode', 'fy'])['Paid']
#     .apply(fy_paid_status)
#     .reset_index()
# )

# # Step 3: Pivot FYs into separate columns
# pivot_df = status_df.pivot(index='consumercode', columns='fy', values='Paid').reset_index()

# print(pivot_df)


In [11]:
# Merge pivot_df with joined_pt_owner
merged_df = joined_pt_owner.merge(
    pivot_df,
    left_on='propertyid_property',  # from joined_pt_owner
    right_on='consumercode',        # from pivot_df
    how='left'
)

# Drop 'consumercode' column from the merge if you don’t need it
merged_df.drop(columns=['consumercode'], inplace=True)

print(merged_df.head())

                                     id propertyid_property     tenantid  \
0  66acb675-bbf5-4a8f-babb-9eb2f9894994     PT-1014-2004760  pb.phagwara   
1  65cb9a53-ede1-4700-a9d9-d62b66b60320     PT-1014-2007961  pb.phagwara   
2  5990622c-c33b-4b84-b84b-2026ddfc742a     PT-1014-2442849  pb.phagwara   
3  059202bd-ef47-4f2f-b9d7-392ea34a1da7      PT-1014-993641  pb.phagwara   
4  059202bd-ef47-4f2f-b9d7-392ea34a1da7      PT-1014-993641  pb.phagwara   

   status          ownershipcategory              usagecategory  landarea  \
0  ACTIVE     INDIVIDUAL.SINGLEOWNER                      MIXED     465.0   
1  ACTIVE     INDIVIDUAL.SINGLEOWNER                RESIDENTIAL     101.3   
2  ACTIVE     INDIVIDUAL.SINGLEOWNER                RESIDENTIAL     204.0   
3  ACTIVE  INDIVIDUAL.MULTIPLEOWNERS  NONRESIDENTIAL.INDUSTRIAL    1640.0   
4  ACTIVE  INDIVIDUAL.MULTIPLEOWNERS  NONRESIDENTIAL.INDUSTRIAL    1640.0   

     createdtime                                  additionaldetails  \
0  172500

In [12]:
print(merged_df.columns)

Index(['id', 'propertyid_property', 'tenantid', 'status', 'ownershipcategory',
       'usagecategory', 'landarea', 'createdtime', 'additionaldetails',
       'propertyid_owner', 'userid', '2013-14', '2014-15', '2015-16',
       '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22',
       '2022-23', '2023-24', '2024-25', '2025-26'],
      dtype='object')


In [13]:
# Identify all FY year columns
fy_cols = [col for col in merged_df.columns if col.startswith("20") and "-" in col]

# Group and aggregate
grouped_df = (
    merged_df
    .groupby(["propertyid_property"] + fy_cols, dropna=False)
    .agg({
        "id" : "first",
        "landarea": "first",               # keep first value from property df
        "usagecategory": "first",           # keep first value from property df
        "userid": lambda x: list(set(x))
    })  # unique list of owners
    .reset_index()
)

# Optional: rename userid column
grouped_df.rename(columns={"userid": "userids_list"}, inplace=True)

print(grouped_df)


      propertyid_property  2013-14  2014-15  2015-16  2016-17  2017-18  \
0          PT-1014-052724      NaN      NaN      NaN      NaN      NaN   
1         PT-1014-1000075      NaN      NaN      NaN      NaN      0.0   
2         PT-1014-1000137      NaN      NaN      NaN      NaN      NaN   
3         PT-1014-1000391      NaN      NaN      NaN      NaN      NaN   
4         PT-1014-1000444      NaN      NaN      NaN      NaN      NaN   
...                   ...      ...      ...      ...      ...      ...   
10186      PT-1014-999670      NaN      NaN      NaN      NaN      NaN   
10187      PT-1014-999710      NaN      NaN      NaN      NaN      NaN   
10188      PT-1014-999750      NaN      NaN      NaN      NaN      NaN   
10189      PT-1014-999756      NaN      NaN      NaN      NaN      NaN   
10190      PT-1014-999823      NaN      NaN      NaN      NaN      NaN   

       2018-19  2019-20  2020-21  2021-22  2022-23  2023-24  2024-25  2025-26  \
0          0.0      NaN      0

In [14]:
print(len(grouped_df))

10191


In [16]:
pt_address_df = pd.read_csv(
    '/home/prerna/Punjab/punjab-data-prod-analysis/phagwara/eg_pt_address_phagwara.csv',
    usecols=['propertyid', 'locality']
)
print(len(pt_address_df))
print(pt_address_df)

40696
                                 propertyid locality
0      9dc74eb1-9dfa-49df-b0f5-a3a3acc4e5da    PLC52
1      a9e2ca6a-4ab3-4d14-9ccf-390056bd7598   PLC114
2      ee87180f-0a8f-491f-9297-f79f9ea7f6e1    PLC98
3      5f5ab5ef-3d51-46cb-9f9c-be58f9ca71a7   PLC154
4      7a98c802-3396-421c-a1ef-27395416e78f   PLC163
...                                     ...      ...
40691  1bdf5005-ffa0-4590-9166-b1197efa2374    PLC74
40692  b4feaa92-ad08-48ad-ba3d-3d908c737b1f   PLC142
40693  44abb2c6-7198-41b7-baba-278eba6eea00   PLC142
40694  1543ba52-87ce-43ae-8e10-aee13b7eec92   PLC131
40695  0fdf2a12-c1e7-4bf2-aa55-cdda42c16eb1   PLC142

[40696 rows x 2 columns]


In [17]:
dup_counts = (
    pt_address_df
    .groupby(['propertyid', 'locality'])
    .size()
    .reset_index(name='count')
    .query('count > 1')  # Keep only duplicates
)

print(dup_counts)
print(f"Total duplicate combinations: {len(dup_counts)}")

                                 propertyid locality  count
5      000fc31c-97c7-417b-bf6b-43aab3c1c026   PLC153      3
8      001237bc-02ec-4429-925b-e60cbd507542    PLC68      3
17     00256c2c-0ca2-4531-a3b1-f69987171d27   PLC180      4
26     003846c2-3141-40be-8158-f3067e138257   PLC123      2
31     00418487-f6fa-492c-bd87-e7e21e028f2e    PLC13      2
...                                     ...      ...    ...
34157  ff5f2699-4ae7-41ae-96b5-ab57244e23ad    PLC13      2
34189  ffa7dbf8-ebc5-4773-9aba-b701f945af26    PLC52      2
34197  ffb76054-4582-454d-b1b8-b71b591c41af   PLC142      2
34216  ffde9b83-0ffd-4c75-ba12-2abe94465041   PLC197      3
34226  fffa6836-6c8b-4cf0-b36f-1ba380222ce0    PLC20      3

[2761 rows x 3 columns]
Total duplicate combinations: 2761


In [18]:
# Collapse duplicates: each propertyid gets a set of localities
address_grouped = (
    pt_address_df
    .groupby('propertyid', dropna=False)['locality']
    .agg(lambda x: list(sorted(set(x.dropna()))))  # unique localities as list
    .reset_index()
)

# Merge back to grouped_df
final_df = grouped_df.merge(
    address_grouped,
    left_on='id',
    right_on='propertyid',
    how='left'
)

# Drop the extra join column
final_df.drop(columns=['propertyid'], inplace=True)

print(final_df.head())
print(len(final_df))

  propertyid_property  2013-14  2014-15  2015-16  2016-17  2017-18  2018-19  \
0      PT-1014-052724      NaN      NaN      NaN      NaN      NaN      0.0   
1     PT-1014-1000075      NaN      NaN      NaN      NaN      0.0      0.0   
2     PT-1014-1000137      NaN      NaN      NaN      NaN      NaN      NaN   
3     PT-1014-1000391      NaN      NaN      NaN      NaN      NaN      NaN   
4     PT-1014-1000444      NaN      NaN      NaN      NaN      NaN      0.0   

   2019-20  2020-21  2021-22  2022-23  2023-24  2024-25  2025-26  \
0      NaN      0.0      0.0      0.0      0.0     0.00      0.0   
1      0.0      0.0    995.0    943.0    882.0   761.00    668.0   
2      NaN      0.0      0.0    276.0    256.0   218.00    188.0   
3      NaN      0.0      NaN      NaN      0.0     0.00      0.0   
4      0.0      0.0      0.0      0.0      0.0     0.18   1644.0   

                                     id  landarea              usagecategory  \
0  34c7e455-d9ce-452f-9c2e-05bafb5f1

In [19]:
fy_cols = [col for col in final_df.columns if col.startswith("20") and "-" in col]
fy_cols_sorted = sorted(fy_cols)  # ensure chronological order

def mark_gaps(row):
    seen_demand = False
    statuses = {}
    for col in fy_cols_sorted:
        val = row[col]
        if pd.notna(val):
            seen_demand = True
            statuses[col + "_status"] = "Demand generated"
        else:
            if seen_demand:
                future_has_demand = any(pd.notna(row[c]) for c in fy_cols_sorted[fy_cols_sorted.index(col)+1:])
                if future_has_demand:
                    statuses[col + "_status"] = "Demand not generated, hence not paid"
                else:
                    statuses[col + "_status"] = "No demand"
            else:
                statuses[col + "_status"] = "No demand"
    return pd.Series(statuses)

status_df = final_df.apply(mark_gaps, axis=1)
merged_with_status = pd.concat([final_df, status_df], axis=1)

print(merged_with_status.head(10))
print(len(merged_with_status))


  propertyid_property  2013-14  2014-15  2015-16  2016-17  2017-18  2018-19  \
0      PT-1014-052724      NaN      NaN      NaN      NaN      NaN      0.0   
1     PT-1014-1000075      NaN      NaN      NaN      NaN      0.0      0.0   
2     PT-1014-1000137      NaN      NaN      NaN      NaN      NaN      NaN   
3     PT-1014-1000391      NaN      NaN      NaN      NaN      NaN      NaN   
4     PT-1014-1000444      NaN      NaN      NaN      NaN      NaN      0.0   
5     PT-1014-1000458      NaN      NaN      NaN      NaN      NaN      0.0   
6     PT-1014-1000465      NaN      NaN      NaN      NaN      NaN      0.0   
7     PT-1014-1001170      NaN      NaN      NaN      NaN      0.0      0.0   
8     PT-1014-1001193      NaN      0.0      0.0      0.0      0.0      0.0   
9     PT-1014-1001307      NaN      NaN      NaN      NaN      NaN      NaN   

   2019-20  2020-21  2021-22  ...    2016-17_status    2017-18_status  \
0      NaN      0.0      0.0  ...         No demand      

In [20]:
import numpy as np

fy_cols = [col for col in final_df.columns if col.startswith("20") and "-" in col]
fy_cols_sorted = sorted(fy_cols)  # ensure chronological order

def fill_gaps(row):
    seen_demand = False
    for col in fy_cols_sorted:
        if pd.notna(row[col]):  
            seen_demand = True
        elif seen_demand:  
            # Only mark NaNs if demand was already seen before this FY
            # and later we might see demand again
            # We'll check forward to confirm
            future_has_demand = any(pd.notna(row[c]) for c in fy_cols_sorted[fy_cols_sorted.index(col)+1:])
            if future_has_demand:
                row[col] = "Demand not generated, hence not paid"
                # row[col] = 0.0
    return row

final_df[fy_cols_sorted] = final_df[fy_cols_sorted].apply(fill_gaps, axis=1)

print(final_df[fy_cols_sorted].head(10))


/tmp/ipykernel_9822/129549214.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Demand not generated, hence not paid' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  row[col] = "Demand not generated, hence not paid"


   2013-14 2014-15 2015-16 2016-17 2017-18 2018-19  \
0      NaN     NaN     NaN     NaN     NaN     0.0   
1      NaN     NaN     NaN     NaN     0.0     0.0   
2      NaN     NaN     NaN     NaN     NaN     NaN   
3      NaN     NaN     NaN     NaN     NaN     NaN   
4      NaN     NaN     NaN     NaN     NaN     0.0   
5      NaN     NaN     NaN     NaN     NaN     0.0   
6      NaN     NaN     NaN     NaN     NaN     0.0   
7      NaN     NaN     NaN     NaN     0.0     0.0   
8      NaN     0.0     0.0     0.0     0.0     0.0   
9      NaN     NaN     NaN     NaN     NaN     NaN   

                                2019-20 2020-21  \
0  Demand not generated, hence not paid     0.0   
1                                   0.0     0.0   
2                                   NaN     0.0   
3                                   NaN     0.0   
4                                   0.0     0.0   
5                                   0.0     0.0   
6                                   0.0     0.0 

In [21]:
print(final_df)

      propertyid_property  2013-14 2014-15 2015-16 2016-17 2017-18 2018-19  \
0          PT-1014-052724      NaN     NaN     NaN     NaN     NaN     0.0   
1         PT-1014-1000075      NaN     NaN     NaN     NaN     0.0     0.0   
2         PT-1014-1000137      NaN     NaN     NaN     NaN     NaN     NaN   
3         PT-1014-1000391      NaN     NaN     NaN     NaN     NaN     NaN   
4         PT-1014-1000444      NaN     NaN     NaN     NaN     NaN     0.0   
...                   ...      ...     ...     ...     ...     ...     ...   
10186      PT-1014-999670      NaN     NaN     NaN     NaN     NaN     0.0   
10187      PT-1014-999710      NaN     NaN     NaN     NaN     NaN     NaN   
10188      PT-1014-999750      NaN     NaN     NaN     NaN     NaN     NaN   
10189      PT-1014-999756      NaN     NaN     NaN     NaN     NaN     NaN   
10190      PT-1014-999823      NaN     NaN     NaN     NaN     NaN     NaN   

                                    2019-20 2020-21  \
0      D

In [23]:
report = final_df.rename(columns={
    'propertyid_property': 'Property ID',
    '2013-14' : '2013-14',
    '2014-15' : '2014-15',
    '2015-16' : '2015-16',
    '2016-17' : '2016-17',
    '2017-18' : '2017-18',
    '2018-19' : '2018-19',
    '2019-20' : '2019-20',
    '2020-21' : '2020-21',
    '2021-22' : '2021-22',
    '2022-23' : '2022-23',
    '2023-24' : '2023-24',
    '2024-25' : '2024-25',
    '2025-26' : '2025-26',
    'landarea' : 'Plot size',
    'usagecategory' : 'Propertyusagetype',
    'locality' : 'Locality',
    'userids_list' : 'Owner Id'
}).copy()

# then keep only those columns, in the exact order you want:
final_report = report[
    [
        'Property ID',
        'Owner Id',
        'Locality',
        'Plot size',
        'Propertyusagetype',
        '2013-14',
        '2014-15',
        '2015-16',
        '2016-17',
        '2017-18',
        '2018-19',
        '2019-20',
        '2020-21',
        '2021-22',
        '2022-23',
        '2023-24',
        '2024-25'
    ]
].copy()

In [24]:
final_report.to_csv('Punjab_Data_Analysis_Phagwara_PDC_FINAL_amountDue_1.csv', index=False)